> objective (loss) vs. gradient

- 修正分布差异：解决“数据复用”**的合法性问题。
    - 标准 Policy Gradient 是 On-Policy 的，要求“边采边练，练完即弃”。这意味着每更新一次参数 $\theta$，之前的经验数据 $\tau$ 就废了，因为 $\pi_\theta$ 变了，数据的分布 $P(\tau|\theta)$ 就变了。这太浪费了！
    - $\mathbb{E}_{x \sim \pi_{new}}[f(x)] = \mathbb{E}_{x \sim \pi_{old}}\left[ \frac{\pi_{new}(x)}{\pi_{old}(x)} \cdot f(x) \right]$
    - 这里的 $\frac{\pi_{new}}{\pi_{old}}$ 就是在强行扭曲 $\pi_{old}$ 的数据分布，让它看起来像是从 $\pi_{new}$ 采出来的。如果不加 Ratio：模型会误以为 $\pi_{old}$ 经常做出的动作，也是 $\pi_{new}$ 经常做出的动作。加上 Ratio：模型会意识到，“哦，虽然这个动作在数据里出现了，但根据我现在的新参数，我其实很少做这个动作（Ratio < 1），所以我计算梯度时要打折，不要被这个过时的数据带偏了。”
    - 它是连接“过去的数据”和“现在的模型”的数学桥梁，保证了梯度的无偏性（Unbiasedness）。
- IS ratio 是 (REINFORCE) 梯度的修正系数
    - Importance Sampling Weight, ISW
    - 这是 Ratio 在优化动力学（Optimization Dynamics）上的作用。它决定了梯度下降时，我们在某个方向上推多大劲。
    - $\nabla J \approx \sum \underbrace{\frac{\pi_{new}}{\pi_{old}}}_{\text{Ratio}} \cdot \underbrace{\hat{A}}_{\text{Advantage}} \cdot \nabla \log \pi$
- 它代表了“信任区域”的边界
    - Ratio 衡量了新策略相对于旧策略在当前动作上“走了多远”：
        - $r_t > 1$：新策略比旧策略更倾向于做这个动作。
        - $r_t < 1$：新策略比旧策略更不倾向于做这个动作。

### PPO

理想 on-policy policy gradient 是：

$$
\nabla J(\theta)
=
\mathbb{E}_{\tau \sim \pi_\theta}
\left[
\nabla_\theta \log \pi_\theta(\tau) R(\tau)
\right]
$$

但是 PPO 不可能每做一次 optimizer update 都重新 rollout。实际做法是先用旧策略采样：

$$
\tau \sim \pi_{\text{old}}
$$

然后在更新新策略时，用 importance sampling ratio 做近似：

$$
r_t(\theta)
=
\frac{
\pi_\theta(a_t \mid s_t)
}{
\pi_{\text{old}}(a_t \mid s_t)
}
$$

PPO clipped surrogate 是：

$$
L^{\text{PPO}}(\theta)
=
\mathbb{E}_{t \sim \pi_{\text{old}}}
\left[
\min
\left(
r_t(\theta) A_t,
\operatorname{clip}
\left(
r_t(\theta),
1-\epsilon,
1+\epsilon
\right)
A_t
\right)
\right]
$$

所以 PPO ratio 本质上就是用 importance sampling 思想，把从旧 policy 采样来的数据近似用于新 policy 的更新。

但 PPO 不是把 off-policy 完全变成 on-policy。它只是一个有 clip 约束的 near-on-policy 近似。

> 经典 PPO 的隐含假设
- 经典 PPO 默认结构是：

```text
用 pi_old rollout
        ↓
得到 trajectory / token 数据
        ↓
固定 pi_old logprob
        ↓
更新 pi_theta
```

因此经典 PPO 隐含假设：

$$
\pi_{\text{rollout}}
=
\pi_{\text{old}}
$$

这时 PPO denominator 同时承担两个角色：

- behavior policy：数据确实由它采样得到。
- trust-region anchor：PPO update 从它开始，不希望当前 policy 离它太远。

也就是说：

$$
r_t(\theta)
=
\frac{
\pi_\theta(a_t \mid s_t)
}{
\pi_{\text{old}}(a_t \mid s_t)
}
$$

里的 $\pi_{\text{old}}$ 既是采样策略，也是 update anchor。

### pg

在标准 Policy Gradient 中，我们的原始目标是最大化期望回报 $J(\theta)$：（$P(\tau|\theta)$ 就是 $\pi_\theta(\tau)$）

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau)] = \sum_{\tau} P(\tau|\theta) R(\tau)$$

这里没有 Log。目标就是“概率 $\times$ 回报”的加权和。当我们对这个 $J(\theta)$ 求梯度 $\nabla J(\theta)$ 时：$$\begin{aligned}
\nabla J(\theta) &= \nabla \sum_{\tau} P(\tau|\theta) R(\tau) \\
&= \sum_{\tau} \nabla P(\tau|\theta) R(\tau) \\
&= \sum_{\tau} P(\tau|\theta) \frac{\nabla P(\tau|\theta)}{P(\tau|\theta)} R(\tau) \quad \text{(引入恒等变换)} \\
&= \sum_{\tau} P(\tau|\theta) \nabla \log P(\tau|\theta) R(\tau) \quad \text{(因为 } \frac{\nabla x}{x} = \nabla \log x \text{)} \\
&= \mathbb{E}_{\tau \sim \pi_\theta} [\nabla \log P(\tau|\theta) R(\tau)]
\end{aligned}$$

REINFORCE 的 $\log$ 是求导结果的一部分，而不是原始目标函数的一部分。

### is

在标准的 Policy Gradient (比如 REINFORCE) 中，梯度是这样的：

$$\nabla J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum \nabla_\theta \log \pi_\theta(a_t|s_t) \hat{A}_t \right]$$

但在 PPO/GRPO 中，我们使用的是 Off-Policy (或者说 On-Policy 的变种) 训练方式。我们拿着 $\pi_{\theta_{old}}$ 采样的数据来更新 $\pi_\theta$。这就引入了一个问题：数据的分布变了。为了解决这个问题，我们利用重要性采样（Importance Sampling）。目标函数（简化版，未截断）是（$J(\theta) = \sum_\tau \pi_{old}(\tau) \frac{\pi_\theta(\tau)}{\pi_{old}(\tau)} R(\tau) = \mathbb{E}_{\tau \sim \pi_{old}} \left[ \frac{\pi_\theta(\tau)}{\pi_{old}(\tau)} R(\tau) \right]$）：

$$L(\theta) = \mathbb{E}_{t \sim \pi_{old}} \left[ \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)} \hat{A}_t \right] = \mathbb{E}_{t \sim \pi_{old}} [ r_t(\theta) \hat{A}_t ]$$

现在，我们对 $\theta$ 求梯度：
$$\begin{aligned}
\nabla_\theta L(\theta) &= \nabla_\theta \left( \frac{\pi_\theta(a|s)}{\pi_{old}(a|s)} \hat{A} \right) \\
&= \frac{\hat{A}}{\pi_{old}(a|s)} \nabla_\theta \pi_\theta(a|s) \quad \text{(因为 $\pi_{old}$ 对 $\theta$ 来说是常数)} \\
\end{aligned}$$

这里利用 log-derivative trick $\nabla \pi = \pi \nabla \log \pi$，代入上式：

$$\begin{aligned}
\nabla_\theta L(\theta) &= \frac{\hat{A}}{\pi_{old}(a|s)} \cdot \left( \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s) \right) \\
&= \underbrace{\frac{\pi_\theta(a|s)}{\pi_{old}(a|s)}}_{\text{Ratio } r_t(\theta)} \cdot \underbrace{\hat{A} \cdot \nabla_\theta \log \pi_\theta(a|s)}_{\text{Standard PG Gradient}}
\end{aligned}$$